# v0.21.0 — Auto-resubscribe & change feeds

Two ways to follow a table in real time, now both resilient:

* **Live queries** (`live()` / `watch()`) push every change the moment it happens. Since v0.21.0 a
  stream **survives a dropped WebSocket**: the ORM reconnects, sends the same live query again and
  the `async for` carries on. `on_reconnect` is where to catch up on what the outage hid, since a
  live query never replays it.
* **Change feeds** (`changes()`, new) read the table's durable log of writes. Nothing is missed
  while nobody reads, and `stream.cursor` resumes **exactly** where a consumer stopped.

Both run on SurrealDB 2.x and 3.x. The two lines number their change feeds differently (2.x's
`SINCE` takes `versionstamp >> 16`, 3.x's the stamp itself); the ORM normalises that, so the same
code resumes correctly on either. Section 6 shows the one visible difference.

## 1. Connect

In [1]:
import asyncio
import contextlib
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")

# Live queries need WebSocket; change feeds would also work over http://.
SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 2. A model, and a table with a change feed

A change feed is opt-in per table: `DEFINE TABLE … CHANGEFEED <retention>`. Only writes made after
it are recorded.

In [2]:
from surreal_orm_lite import BaseSurrealModel


class Invoice(BaseSurrealModel):
    id: str
    status: str = "draft"
    total: float = 0.0


client = await SurrealDBConnectionManager.get_client()
with contextlib.suppress(Exception):
    await client.query("REMOVE TABLE Invoice;", {})
await client.query("DEFINE TABLE Invoice SCHEMALESS CHANGEFEED 1h;", {})
print("Table Invoice ready, with a 1h change feed")


async def take(stream, count, timeout=5.0):
    """Read `count` events, failing instead of hanging if they never come."""
    events = []
    async with asyncio.timeout(timeout):
        while len(events) < count:
            events.append(await anext(stream))
    return events

Table Invoice ready, with a 1h change feed


## 3. `changes()` — reading the feed

`start()` fixes "now" (the default `since=None`) before the writes below; without it, "now" would
be the first read. Each event is the same `ModelChangeEvent` a live query yields. A change feed
records writes, not their kind, so a creation arrives as `UPDATE`.

In [3]:
stream = await Invoice.objects().changes(poll_interval=0.05).start()

inv1 = Invoice(id="inv1", total=100.0)
await inv1.save()
await inv1.merge(status="sent")
await Invoice(id="inv2", total=40.0).save()

for event in await take(stream, 3):
    print(event.action, event.record_id, event.instance.status, event.instance.total)

UPDATE Invoice:inv1 draft 100.0
UPDATE Invoice:inv1 sent 100.0
UPDATE Invoice:inv2 draft 40.0


## 4. Resuming exactly from the cursor

`stream.cursor` is an `int`. Save it after processing each event; a consumer that restarts passes it
back as `since=` and gets every later change — none skipped, none repeated.

In [4]:
cursor = stream.cursor
stream.stop()
print("cursor is an", type(cursor).__name__)

# The consumer is down; writes keep happening.
await Invoice(id="inv3", total=7.5).save()
await inv1.delete()

resumed = Invoice.objects().changes(since=cursor, poll_interval=0.05)
for event in await take(resumed, 2):
    print(event.action, event.record_id)

try:
    await take(resumed, 1, timeout=0.5)
except TimeoutError:
    print("nothing else: no duplicate, no gap")
resumed.stop()

cursor is an int
UPDATE Invoice:inv3
DELETE Invoice:inv1


nothing else: no duplicate, no gap


## 5. A transaction is delivered whole

The cursor moves past a transaction only once its **last** event has been yielded. A consumer that
crashes halfway through gets the whole transaction again on resume — at-least-once, never half of
it.

In [5]:
batch = await Invoice.objects().changes(poll_interval=0.05).start()
before = batch.cursor
await client.query("BEGIN; CREATE Invoice:a SET total = 1; CREATE Invoice:b SET total = 2; COMMIT;", {})

await take(batch, 1)
print("after the 1st event of the transaction, cursor moved:", batch.cursor != before)
await take(batch, 1)
print("after the 2nd (last) event, cursor moved:", batch.cursor != before)
batch.stop()

after the 1st event of the transaction, cursor moved: False
after the 2nd (last) event, cursor moved: True


## 6. `since=` a moment in time

SurrealDB 3.x answers `SINCE <datetime>` with nothing at all, so the ORM converts the moment to the
server's own versionstamp and the result is **exact**. On 2.x the server's datetime lookup is coarse
and also returns some earlier changes, so it is **at-least-once**.

In [6]:
moment = await client.query("RETURN time::now();", {})
await asyncio.sleep(0.1)
await Invoice(id="late", total=3.0).save()

since_moment = Invoice.objects().changes(since=moment, poll_interval=0.05)
seen = []
async with asyncio.timeout(5):
    while "Invoice:late" not in seen:
        seen.append((await anext(since_moment)).record_id)
since_moment.stop()

if seen == ["Invoice:late"]:
    print("exactly the change made after the moment")
else:
    print(f"{len(seen)} changes, ending with the late one (SurrealDB 2.x: at-least-once)")

exactly the change made after the moment


## 7. A live query that survives a dropped connection

The cell closes the WebSocket under the ORM's feet, which is what a network failure, a server
restart or an idle proxy timeout looks like from the client. The stream reconnects with backoff,
`on_reconnect` runs with the old and new live-query ids, and a write made afterwards arrives as
usual.

In [7]:
reconnected = asyncio.Event()


async def on_reconnect(old_id, new_id):
    print("on_reconnect: new subscription id:", old_id != new_id)
    # The place to catch up: a live query does not replay what changed during the outage.
    reconnected.set()


async with Invoice.objects().filter(status="overdue").live(on_reconnect=on_reconnect) as live:
    first_id = live.live_id

    current = await SurrealDBConnectionManager.get_client()
    await current.socket.close()  # simulate the network dropping the connection

    async with asyncio.timeout(10):
        await reconnected.wait()

    await Invoice(id="inv9", status="draft", total=1.0).save()  # filtered out
    await Invoice(id="inv10", status="overdue", total=2.0).save()
    event = await take(live, 1)
    print(event[0].action, event[0].record_id, "| live_id changed:", live.live_id != first_id)

on_reconnect: new subscription id: True
CREATE Invoice:inv10 | live_id changed: True


## 8. Opting out: `auto_resubscribe=False`

The stream then ends on a drop with `SurrealDbConnectionError`, instead of waiting forever as it did
before v0.21.0. The same applies to secondary `subscribe_live()` readers. A teardown you ask for
(`kill()`, `close_connection()`, leaving the block) never triggers a resubscribe.

In [8]:
from surreal_orm_lite import SurrealDbConnectionError

async with Invoice.objects().live(auto_resubscribe=False) as fragile:
    current = await SurrealDBConnectionManager.get_client()
    await current.socket.close()
    try:
        await take(fragile, 1)
    except SurrealDbConnectionError:
        print("SurrealDbConnectionError raised; is_active:", fragile.is_active)

SurrealDbConnectionError raised; is_active: False


## 9. What `changes()` refuses

`SHOW CHANGES` has no `WHERE`: a filtered queryset would silently stream the whole table, so it is
refused by name. A table without a change feed would answer with nothing forever, so it is refused
too, with the `DEFINE TABLE` to run.

In [9]:
from surreal_orm_lite import SurrealDbError

try:
    Invoice.objects().filter(status="sent").changes()
except SurrealDbError as exc:
    print("refused:", str(exc).split(":")[0])

client = await SurrealDBConnectionManager.get_client()
await client.query("DEFINE TABLE OVERWRITE NoFeed SCHEMALESS;", {})


class NoFeed(BaseSurrealModel):
    id: str


try:
    await NoFeed.objects().changes().start()
except SurrealDbError as exc:
    print("refused:", str(exc).split(",")[0])

refused: A change feed cannot honour filter()


refused: Table 'NoFeed' has no change feed


## 10. Cleanup

In [10]:
client = await SurrealDBConnectionManager.get_client()
for table in ("Invoice", "NoFeed"):
    with contextlib.suppress(Exception):
        await client.query(f"REMOVE TABLE {table};", {})
await SurrealDBConnectionManager.close_connection()
print("done")

done
